# booth-gpu — GPU vs CPU benchmark (Colab, free T4)

Re-expresses the booth-detection hot path (color mask + connected-components + dedup/merge) as **pure GPU tensor ops** and measures the real speedup against the production CPU code.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.

Then `Runtime → Run all`. The notebook clones the repo, installs deps, builds fixtures, runs the benchmark, and runs the parity check.

In [ ]:
# 0. Confirm we actually have a GPU
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! No GPU. Runtime > Change runtime type > GPU (T4), then Run all again.')
!nvidia-smi -L 2>/dev/null || true

In [ ]:
# 1. Clone the repo
import os
REPO_URL = 'https://github.com/mayanksoninls161-beep/booth-gpu.git'
if not os.path.isdir('booth-gpu'):
    !git clone $REPO_URL
%cd booth-gpu
!git pull   # make sure we have the latest pushed fixes
!ls -1 src

In [ ]:
# 2. Deps. Colab already ships a CUDA torch; we only need headless OpenCV
#    for the CPU reference (cv2.inRange / connectedComponents / intersectConvexConvex).
!pip -q install opencv-python-headless

In [ ]:
# 3. Build fixtures (synthetic dense plan: ~600 true cells -> ~2k box pool)
!python src/make_fixtures.py --n-cells 600

In [ ]:
# 4. THE BENCHMARK: CPU vs GPU, with parity, for merge + mask/components
!python src/bench.py --n-cells 600

In [ ]:
# 5. Parity check (GPU output == production CPU output within tolerance)
!python tests/test_parity.py

## Use the REAL pre-NMS pool (optional)

The synthetic pool mimics a dense plan's structure. To benchmark on the *actual* box pool your pipeline produces:

1. Inside the container, copy `src/export_real_fixtures.py` to `/app`, run it in front of one dense-plan detection (see the file's docstring), and copy the resulting `boxes_pool_real.json` into `fixtures/` here.
2. Then:

```python
!python src/bench.py --pool fixtures/boxes_pool_real.json
```

The fixture is bbox+score only — no source pixels, no secrets.

In [ ]:
# 6. (optional) bigger pool to see how the GPU gap widens with N
!python src/bench.py --n-cells 1200 --no-image

## Try it on YOUR real plan (image or PDF)

Upload a floor-plan **image (png/jpg)** or **PDF**, run the full GPU pipeline on it,
and view the annotated result inline.

**Pick the detector for your plan (`--mode`):**

- `--mode bordered` *(best for dense exhibition plans)* — keys on the dark **grid
  lines**, so each booth is the cell *enclosed* by them. Every booth is found —
  pale, white, or coloured — and two touching booths stay separate because the dark
  line between them is background. Run it **full-image** (`--tile 0`): bordered
  detection needs no tiling, so it skips the seam-clip + global-merge steps that can
  over-merge a dense plan.
- `--mode color` — keys on the fill **colour** (auto-detected dominant hues). Best
  for plans whose booths are big solid colour blocks. It can fuse same-colour
  neighbours and misses pale/white cells, so it under-counts packed grids.
- `--mode both` *(default)* — pools both detectors; the merge's merged-block
  pre-filter then drops any coarse colour blob that the fine bordered cells tile.

Other knobs:
- PDFs rasterise at `--dpi` (default 200); pick a page with `--page`.
- `--min-area` drops tiny noise; `--line-thresh` / `--seal-ksize` tune the bordered pass.
- `--merge assisted` (default) is exact greedy NMS. `--merge cluster` is faster but
  collapses a whole chain of overlapping boxes into one — it under-counts dense plans.
- Big plans: add `--downscale 0.5` if it's slow.
- Nothing leaves the Colab VM.

In [ ]:
# 7a. PDF support (only needed if you upload a .pdf). Images need nothing extra.
!pip -q install PyMuPDF

In [ ]:
# 7b. Upload your plan (image or PDF), run the GPU pipeline, show the result.
from google.colab import files
from IPython.display import Image, display

uploaded = files.upload()                       # pick one image or PDF
src_path = next(iter(uploaded))                  # the uploaded filename
print('uploaded:', src_path)

# DENSE BORDERED plans (packed exhibition grids): detect the cells ENCLOSED by the
# grid lines, full-image. No tiling -> no seam-clip and no chain-collapsing merge,
# which is what recovers the most booths on a dense plan.
!python src/run_real.py --input "$src_path" --mode bordered --tile 0 --min-area 300

# Alternatives:
#   colour-block plans:  --mode color --tile 1800 --overlap 400
#   mixed / unsure:      --mode both  --tile 0          (default mode)
#   if it is slow:       add --downscale 0.5

stem = src_path.rsplit('.', 1)[0]
print('\n--- detected boxes ---'); display(Image(f'out/{stem}_boxes.png'))
print('--- preview mask (grid lines) ---'); display(Image(f'out/{stem}_mask.png'))